In [ ]:
# ============================================================
# SYNTHETIC CROWD GENERATOR V7
# ============================================================
#
# COMBINES
# ------------------------------------------------------------
# ✔ Explicit temporal phases (bottleneck + panic)
# ✔ Expanding panic wave from persistent origin
# ✔ Stress memory accumulation per agent
# ✔ Stop-go bottleneck oscillations
# ✔ Queue-front compression propagation
# ✔ post_warmup_factor ramp (Fix 4) — weakens early signal
#   so LSTM must accumulate evidence over time
# ✔ Per-sequence phase offset for normal sequences
# ✔ Group boundary reflection fix
# ✔ Group index stored (not reference)
# ✔ panic_origin only for panic class
# ✔ Bounded global flow magnitudes
# ✔ Identical agent priors at spawn
# ✔ 200 sequences per class
#
# ============================================================

import os
import cv2
import numpy as np
import scipy.ndimage
from scipy.spatial import KDTree

# ============================================================
# CONFIG
# ============================================================

IMG_SIZE       = 512
FRAMES_PER_SEQ = 30

NUM_NORMAL     = 200
NUM_BOTTLENECK = 200
NUM_PANIC      = 200

BASE_DIR      = "synthetic_crowd_v7"
WARMUP_FRAMES = 5

np.random.seed(42)

CLASS_NAMES = {
    0: "0_normal",
    1: "1_bottleneck",
    2: "2_panic"
}

for c in CLASS_NAMES.values():
    os.makedirs(os.path.join(BASE_DIR, c), exist_ok=True)

# ============================================================
# GEOMETRY
# ============================================================

def random_geometry():

    wall_y         = np.random.randint(380, 440)
    wall_thickness = np.random.randint(50, 90)
    gate_width     = np.random.randint(40, 80)
    gate_x         = np.random.randint(80, IMG_SIZE - 80 - gate_width)

    return {
        "wall_y_min":  wall_y,
        "wall_y_max":  wall_y + wall_thickness,
        "gate_x_min":  gate_x,
        "gate_x_max":  gate_x + gate_width,
        "gate_center": np.array([
            gate_x + gate_width / 2,
            wall_y + wall_thickness / 2
        ], dtype=np.float32)
    }

# ============================================================
# COLLISION
# ============================================================

def circle_rect_collision(cx, cy, r, rx1, ry1, rx2, ry2):
    nearest_x = np.clip(cx, rx1, rx2)
    nearest_y = np.clip(cy, ry1, ry2)
    dx = cx - nearest_x
    dy = cy - nearest_y
    return (dx * dx + dy * dy) < (r * r)

def wall_collision(pos, r, g):
    x, y = pos
    if circle_rect_collision(x, y, r, 0, g["wall_y_min"],
                             g["gate_x_min"], g["wall_y_max"]):
        return True
    if circle_rect_collision(x, y, r, g["gate_x_max"], g["wall_y_min"],
                             IMG_SIZE, g["wall_y_max"]):
        return True
    return False

# ============================================================
# GROUP
# ============================================================

class CrowdGroup:

    def __init__(self, geometry):

        self.center = np.array([
            np.random.uniform(50, IMG_SIZE - 50),
            np.random.uniform(50, geometry["wall_y_min"] - 50)
        ], dtype=np.float32)

        self.velocity = np.random.randn(2).astype(np.float32)
        self.velocity /= (np.linalg.norm(self.velocity) + 1e-6)

        self.coherence = np.random.uniform(0.3, 1.0)
        self.radius    = np.random.uniform(40, 120)
        self.timer     = np.random.randint(20, 80)

    def update(self, geometry):

        self.timer -= 1

        if self.timer <= 0:
            self.velocity = np.random.randn(2).astype(np.float32)
            self.velocity /= (np.linalg.norm(self.velocity) + 1e-6)
            self.coherence = np.random.uniform(0.3, 1.0)
            self.timer     = np.random.randint(20, 80)

        # Reflect on boundary contact so groups don't
        # get stuck at walls with outward-pointing velocity
        new_center = self.center + self.velocity * 1.5
        if new_center[0] < 40 or new_center[0] > IMG_SIZE - 40:
            self.velocity[0] *= -1
        if new_center[1] < 40 or new_center[1] > geometry["wall_y_min"] - 40:
            self.velocity[1] *= -1

        self.center += self.velocity * 1.5
        self.center[0] = np.clip(self.center[0], 40, IMG_SIZE - 40)
        self.center[1] = np.clip(self.center[1], 40,
                                  geometry["wall_y_min"] - 40)

# ============================================================
# AGENT
# ============================================================

class Agent:

    def __init__(self, geometry, num_groups):

        self.r = np.random.uniform(5, 9)

        while True:
            x = np.random.uniform(self.r, IMG_SIZE - self.r)
            y = np.random.uniform(self.r, geometry["wall_y_min"] - 60)
            if not wall_collision((x, y), self.r, geometry):
                break

        self.pos  = np.array([x, y], dtype=np.float32)
        self.prev = self.pos.copy()
        self.vel  = np.zeros(2, dtype=np.float32)

        # Store index, not reference — avoids stale reference bugs
        self.group_idx   = np.random.randint(0, num_groups)
        self.group_timer = np.random.randint(20, 60)

        # Identical priors at spawn — no class signal before warmup
        self.goal_strength   = np.random.uniform(0.8, 1.5)
        self.noise_strength  = np.random.uniform(0.3, 0.8)
        self.social_strength = np.random.uniform(0.5, 1.5)

        self.role = np.random.choice(
            ["gate", "wander", "group"],
            p=[0.4, 0.3, 0.3]
        )

        self.local_target = np.array([
            np.random.uniform(0, IMG_SIZE),
            np.random.uniform(0, geometry["wall_y_min"])
        ], dtype=np.float32)

        # Persistent internal state — accumulates from crowd pressure
        # Starts at a small random value identical across all classes
        self.stress  = np.random.uniform(0.0, 0.2)
        self.fatigue = 0.0

# ============================================================
# TEMPORAL PHASES
# ============================================================

def get_bottleneck_phase(t):
    """
    Maps frame index to a named bottleneck phase.
    Phases are irreversible — crowd cannot un-congest.
    This creates a causal temporal signature that temporal
    shuffle destroys, forcing the LSTM to use ordering.
    """
    if   t < 10: return "converging"
    elif t < 17: return "compression"
    elif t < 23: return "stop_go"
    else:        return "queue_lock"

def get_panic_phase(t):
    """
    Maps frame index to a named panic phase.
    Panic escalates irreversibly from localised disturbance
    to global fragmentation over the sequence.
    """
    if   t < 10: return "disturbance"
    elif t < 15: return "localized_escape"
    elif t < 20: return "propagation"
    elif t < 25: return "fragmentation"
    else:        return "global_panic"

# ============================================================
# GLOBAL FLOW FIELD
# ============================================================

def compute_global_flow(effective_class, geometry, t, seq_phase):
    """
    Scene-level flow field that biases all agents.
    Each class has a distinct temporal evolution.
    seq_phase ensures normal sequences are not all identical.
    """

    temporal_factor = t / FRAMES_PER_SEQ

    # post_warmup_factor: 0.0 at frame 5, 1.0 at frame 29
    # Used for class-specific scaling so early post-warmup
    # frames are still ambiguous and the class signal builds
    # gradually — forcing the LSTM to accumulate evidence
    post_warmup = max(0.0,
        (t - WARMUP_FRAMES) / (FRAMES_PER_SEQ - WARMUP_FRAMES)
    )

    if effective_class == 0:

        # Normal: gentle oscillating field, unique phase per sequence
        angle = np.sin(temporal_factor * np.pi * 2 + seq_phase) * 0.5
        return np.array([np.cos(angle), np.sin(angle)], dtype=np.float32)

    elif effective_class == 1:

        # Bottleneck: convergence toward gate grows quadratically
        # with post_warmup so early frames are still ambiguous
        gate_dir = geometry["gate_center"] - np.array(
            [IMG_SIZE / 2, geometry["wall_y_min"] / 2]
        )
        gate_dir /= (np.linalg.norm(gate_dir) + 1e-6)
        strength = post_warmup ** 2   # quadratic ramp from warmup end
        return gate_dir * strength

    else:

        # Panic: random direction each frame, magnitude grows
        # Capped at 1.5 — previous 2.5 caused all agents to
        # hit speed limit every frame, making speed a trivial feature
        angle     = np.random.uniform(0, np.pi * 2)
        magnitude = np.clip(0.3 + post_warmup * 1.2, 0.3, 1.5)
        return np.array(
            [np.cos(angle), np.sin(angle)], dtype=np.float32
        ) * magnitude

# ============================================================
# UPDATE AGENTS
# ============================================================

def update_agents(
    agents, groups, geometry,
    class_type, t,
    panic_origin, seq_phase
):
    temporal_factor = t / FRAMES_PER_SEQ

    # post_warmup_factor: 0 at frame 5, 1 at frame 29
    # Scales all class-specific forces so they build gradually
    # after warmup ends rather than switching on abruptly
    post_warmup = max(0.0,
        (t - WARMUP_FRAMES) / (FRAMES_PER_SEQ - WARMUP_FRAMES)
    )

    # Warmup: all classes identical
    effective_class = 0 if t < WARMUP_FRAMES else class_type

    # Determine phase names (only used for the relevant class)
    bottleneck_phase = get_bottleneck_phase(t) if effective_class == 1 else None
    panic_phase      = get_panic_phase(t)      if effective_class == 2 else None

    # Panic wave radius expands with phase
    panic_radius_map = {
        "disturbance":     40,
        "localized_escape": 80,
        "propagation":     140,
        "fragmentation":   220,
        "global_panic":    400
    }
    panic_radius = panic_radius_map.get(panic_phase, 0)

    for g in groups:
        g.update(geometry)

    positions = np.array([a.pos for a in agents])
    tree      = KDTree(positions)

    global_flow = compute_global_flow(
        effective_class, geometry, t, seq_phase
    )

    for a in agents:

        a.prev = a.pos.copy()

        # Group switching
        a.group_timer -= 1
        if a.group_timer <= 0:
            a.group_idx   = np.random.randint(0, len(groups))
            a.group_timer = np.random.randint(20, 60)
        current_group = groups[a.group_idx]

        # Role target
        if a.role == "gate":
            target = geometry["gate_center"]
        elif a.role == "wander":
            if np.random.rand() < 0.02:
                a.local_target = np.array([
                    np.random.uniform(0, IMG_SIZE),
                    np.random.uniform(0, geometry["wall_y_min"])
                ], dtype=np.float32)
            target = a.local_target
        else:
            target = current_group.center

        # Goal force
        goal = target - a.pos
        goal /= (np.linalg.norm(goal) + 1e-6)

        # Neighbors
        neighbor_idx  = tree.query_ball_point(a.pos, 35)
        local_density = max(0, len(neighbor_idx) - 1)

        # Repulsion
        repulsion = np.zeros(2, dtype=np.float32)
        for idx in neighbor_idx:
            other = agents[idx]
            if other is a:
                continue
            delta   = a.pos - other.pos
            dist    = np.linalg.norm(delta) + 1e-6
            overlap = (a.r + other.r) - dist
            if overlap > 0:
                repulsion += (delta / dist) * overlap * 1.0

        # Alignment with neighbours
        alignment = np.zeros(2, dtype=np.float32)
        if len(neighbor_idx) > 1:
            for idx in neighbor_idx:
                other = agents[idx]
                if other is not a:
                    alignment += other.vel
            alignment /= (len(neighbor_idx) - 1)

        # Group coherence — scaled by per-agent social_strength
        group_force  = current_group.center - a.pos
        group_force /= (np.linalg.norm(group_force) + 1e-6)
        group_force *= current_group.coherence * a.social_strength

        # Local pressure
        pressure = local_density / 10.0

        # Stress memory — accumulates from crowd pressure, decays slowly
        # This gives agents history that changes their behaviour over time
        a.stress += pressure * 0.01
        a.stress *= 0.995
        a.stress  = np.clip(a.stress, 0, 3)

        # Panic contagion from persistent origin point
        # Only active for panic class, only if agent is inside panic_radius
        panic_force = np.zeros(2, dtype=np.float32)
        if effective_class == 2 and panic_origin is not None and panic_radius > 0:
            delta = a.pos - panic_origin
            dist  = np.linalg.norm(delta) + 1e-6
            if dist < panic_radius:
                panic_force += (delta / dist) * (panic_radius - dist) * 0.03
                a.stress    += 0.03

        # ====================================================
        # CLASS + PHASE SPECIFIC DYNAMICS
        # All forces scaled by post_warmup so they ramp in
        # gradually after warmup ends rather than abruptly.
        # This is Fix 4 — makes early windows ambiguous and
        # forces the LSTM to accumulate temporal evidence.
        # ====================================================

        if effective_class == 0:

            goal_scale   = 0.6
            align_scale  = 0.5
            noise_scale  = 0.4 + 0.2 * post_warmup
            global_scale = 0.4
            speed_limit  = 2.0 + np.random.uniform(-0.2, 0.2)

        elif effective_class == 1:

            if bottleneck_phase == "converging":
                goal_scale   = (0.5 + 0.5 * post_warmup)
                align_scale  = min(0.8, 0.4 + 0.6 * post_warmup)
                noise_scale  = 0.5
                global_scale = (0.3 + 0.7 * post_warmup)
                speed_limit  = 2.2

            elif bottleneck_phase == "compression":
                goal_scale   = (0.8 + 0.5 * post_warmup)
                align_scale  = min(1.0, 0.6 + 0.6 * post_warmup)
                noise_scale  = 0.3
                global_scale = (0.7 + 0.6 * post_warmup)
                speed_limit  = max(0.5, 1.8 - pressure * 0.3)

            elif bottleneck_phase == "stop_go":
                oscillation  = 0.7 + 0.3 * np.sin(t * 0.8)
                goal_scale   = 1.2
                align_scale  = 1.0
                noise_scale  = 0.3
                global_scale = 1.5
                speed_limit  = max(0.3, 2.2 * oscillation - pressure * 0.2)

            else:  # queue_lock
                goal_scale   = 1.5
                align_scale  = min(1.3, 1.0 + 0.3 * post_warmup)
                noise_scale  = 0.2
                global_scale = 1.8
                speed_limit  = max(0.3, 1.0 - pressure * 0.1)

        else:  # panic

            if panic_phase == "disturbance":
                noise_scale  = 0.5 + 0.3 * post_warmup
                global_scale = 0.3 + 0.3 * post_warmup
                align_scale  = 0.5
                speed_limit  = 2.0 + 0.5 * post_warmup

            elif panic_phase == "localized_escape":
                noise_scale  = 1.0 + 0.3 * post_warmup
                global_scale = 0.8 + 0.2 * post_warmup
                align_scale  = 0.3
                speed_limit  = 2.8 + 0.4 * post_warmup

            elif panic_phase == "propagation":
                noise_scale  = 1.5 + 0.3 * post_warmup
                global_scale = 1.2 + 0.3 * post_warmup
                align_scale  = 0.2
                speed_limit  = 3.5 + 0.3 * post_warmup

            elif panic_phase == "fragmentation":
                noise_scale  = 2.0 + 0.2 * post_warmup
                global_scale = 1.6 + 0.2 * post_warmup
                align_scale  = 0.1
                speed_limit  = 4.0 + 0.2 * post_warmup

            else:  # global_panic
                noise_scale  = 2.5
                global_scale = 2.0
                align_scale  = 0.05
                speed_limit  = min(5.0, 4.5 + pressure * 0.1)

            goal_scale = 0.3

        turbulence = np.random.randn(2) * noise_scale

        force = (
            goal        * goal_scale
            + repulsion  * 2.5
            + alignment  * align_scale
            + group_force                  # already scaled by social_strength
            + global_flow * global_scale
            + turbulence
            + panic_force
        )

        # Queue front compression — slows agents near gate
        # in bottleneck, scaled by post_warmup so effect builds
        if effective_class == 1:
            dist_gate = np.linalg.norm(a.pos - geometry["gate_center"])
            if dist_gate < 140:
                compression = pressure * post_warmup
                force *= (1.0 - compression * 0.2)

        # Speed cap
        speed = np.linalg.norm(force)
        if speed > speed_limit:
            force = (force / speed) * speed_limit

        proposed    = a.pos + force
        proposed[0] = np.clip(proposed[0], a.r, IMG_SIZE - a.r)
        proposed[1] = np.clip(proposed[1], a.r, IMG_SIZE - a.r)

        if not wall_collision(proposed, a.r, geometry):
            a.vel = proposed - a.pos
            a.pos = proposed
        else:
            a.vel *= -0.2

# ============================================================
# DENSITY MAP
# ============================================================

def generate_density(agents):
    density = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for a in agents:
        x = int(a.pos[0])
        y = int(a.pos[1])
        if 0 <= x < IMG_SIZE and 0 <= y < IMG_SIZE:
            density[y, x] += 1.0
    return scipy.ndimage.gaussian_filter(density, sigma=5)

# ============================================================
# FLOW MAP
# ============================================================

def generate_flow(agents):
    flow = np.zeros((IMG_SIZE, IMG_SIZE, 2), dtype=np.float32)
    for a in agents:
        x = int(a.pos[0])
        y = int(a.pos[1])
        if 0 <= x < IMG_SIZE and 0 <= y < IMG_SIZE:
            flow[y, x, 0] = a.vel[0]
            flow[y, x, 1] = a.vel[1]
    return flow

# ============================================================
# RENDER
# ============================================================

def render_frame(agents, geometry):

    bg  = np.random.randint(235, 256)
    img = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.uint8) * bg

    wall_color = np.random.randint(90, 160)

    cv2.rectangle(img, (0, geometry["wall_y_min"]),
                  (geometry["gate_x_min"], geometry["wall_y_max"]),
                  wall_color, -1)
    cv2.rectangle(img, (geometry["gate_x_max"], geometry["wall_y_min"]),
                  (IMG_SIZE, geometry["wall_y_max"]),
                  wall_color, -1)

    for a in agents:
        intensity = np.random.randint(0, 50)
        cv2.circle(img, (int(a.pos[0]), int(a.pos[1])),
                   int(a.r), intensity, -1)

    if np.random.rand() < 0.3:
        k   = np.random.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)

    if np.random.rand() < 0.3:
        noise = np.random.normal(0, 5, img.shape)
        img   = np.clip(img + noise, 0, 255).astype(np.uint8)

    return img

# ============================================================
# SEQUENCE
# ============================================================

def generate_sequence(seq_id, class_type):

    geometry = random_geometry()

    seq_dir = os.path.join(
        BASE_DIR, CLASS_NAMES[class_type], f"seq_{seq_id:04d}"
    )
    os.makedirs(seq_dir, exist_ok=True)

    num_agents = np.random.randint(80, 160)
    num_groups = np.random.randint(4, 10)

    groups = [CrowdGroup(geometry) for _ in range(num_groups)]
    agents = [Agent(geometry, num_groups) for _ in range(num_agents)]

    # Panic origin: fixed per sequence, only meaningful for panic class
    # Passed to update_agents for all classes but only applied
    # when effective_class == 2
    panic_origin = np.array([
        np.random.uniform(100, IMG_SIZE - 100),
        np.random.uniform(100, geometry["wall_y_min"] - 100)
    ], dtype=np.float32) if class_type == 2 else None

    # Per-sequence phase offset for normal flow field
    # Ensures no two normal sequences have identical flow directions
    seq_phase = np.random.uniform(0, np.pi * 2)

    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    mask[
        geometry["wall_y_min"]:geometry["wall_y_max"],
        geometry["gate_x_min"]:geometry["gate_x_max"]
    ] = 1
    np.save(os.path.join(seq_dir, "mask.npy"), mask)

    for t in range(FRAMES_PER_SEQ):

        update_agents(
            agents, groups, geometry,
            class_type, t,
            panic_origin, seq_phase
        )

        frame   = render_frame(agents, geometry)
        density = generate_density(agents)
        flow    = generate_flow(agents)

        cv2.imwrite(os.path.join(seq_dir, f"frame_{t:03d}.png"), frame)
        np.save(os.path.join(seq_dir, f"density_{t:03d}.npy"), density)
        np.save(os.path.join(seq_dir, f"flow_{t:03d}.npy"), flow)

# ============================================================
# DRIVER
# ============================================================

seq_id = 0

print("Generating NORMAL sequences...")
for _ in range(NUM_NORMAL):
    generate_sequence(seq_id, 0)
    seq_id += 1

print("Generating BOTTLENECK sequences...")
for _ in range(NUM_BOTTLENECK):
    generate_sequence(seq_id, 1)
    seq_id += 1

print("Generating PANIC sequences...")
for _ in range(NUM_PANIC):
    generate_sequence(seq_id, 2)
    seq_id += 1

print("=" * 60)
print("SYNTHETIC DATASET GENERATION COMPLETE")
print("=" * 60)

In [ ]:
#revised data split
import os
import shutil
import random

BASE_DIR = "synthetic_crowd_v7"
SPLIT_BASE = "split_dataset_v7"
CLASSES = ["0_normal", "1_bottleneck", "2_panic"]

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(42)

if os.path.exists(SPLIT_BASE):
    print(f"Removing old split: {SPLIT_BASE}")
    shutil.rmtree(SPLIT_BASE)

for split in ['train', 'val', 'test']:
    for c in CLASSES:
        os.makedirs(os.path.join(SPLIT_BASE, split, c), exist_ok=True)

print("\nSplitting dataset...")

for c in CLASSES:

    class_path = os.path.join(BASE_DIR, c)

    sequences = [
        s for s in sorted(os.listdir(class_path))
        if os.path.isdir(os.path.join(class_path, s))
    ]

    random.shuffle(sequences)

    n_train = int(len(sequences) * TRAIN_RATIO)
    n_val   = int(len(sequences) * VAL_RATIO)

    splits = {
        'train': sequences[:n_train],
        'val':   sequences[n_train : n_train + n_val],
        'test':  sequences[n_train + n_val:]
    }

    for split_name, seq_list in splits.items():
        for seq in seq_list:
            src = os.path.join(class_path, seq)
            # FIX: c and split_name are used directly here,
            # not captured inside a nested function.
            dst = os.path.join(SPLIT_BASE, split_name, c, seq)
            shutil.copytree(src, dst)

    print(f"\nClass : {c}")
    print(f"  Total : {len(sequences)}")
    print(f"  Train : {len(splits['train'])}")
    print(f"  Val   : {len(splits['val'])}")
    print(f"  Test  : {len(splits['test'])}")

print("\n" + "="*60)
print("SPLIT COMPLETE")
print("="*60)

In [1]:
# ============================================================
# CELL 1 : IMPORTS + DEVICE SETUP + ABLATION CONFIG
# ============================================================

import os
import glob
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ============================================================
# ABLATION CONFIGURATION — SET THIS BEFORE RUNNING
# ============================================================
#
# Four configurations to run independently, one per notebook
# execution (restart kernel between runs to avoid state leakage):
#
#   "baseline"              : strided flow_reduce, NO augmentation
#   "baseline_aug"          : strided flow_reduce, WITH augmentation
#   "motioncnn_no_aug"      : MotionCNN, NO augmentation
#   "motioncnn_aug"         : MotionCNN, WITH augmentation (full model)
#
# This single flag controls both Cell 2 (augmentation on/off)
# and Cell 3 (flow branch architecture selection), so the two
# cells always stay consistent with each other.
#
# ============================================================

ABLATION_CONFIG = "motioncnn_no_aug"   # <-- CHANGE THIS PER RUN

assert ABLATION_CONFIG in [
    "baseline", "baseline_aug", "motioncnn_no_aug", "motioncnn_aug"
], "Invalid ABLATION_CONFIG value"

USE_MOTIONCNN  = ABLATION_CONFIG in ["motioncnn_no_aug", "motioncnn_aug"]
USE_AUGMENTATION = ABLATION_CONFIG in ["baseline_aug", "motioncnn_aug"]

CHECKPOINT_NAME = f"best_crowd_model_{ABLATION_CONFIG}.pth"

print("=" * 60)
print("ABLATION CONFIGURATION")
print("=" * 60)
print(f"Config name       : {ABLATION_CONFIG}")
print(f"Flow branch       : {'MotionCNN (deep, SE attention)' if USE_MOTIONCNN else 'Strided flow_reduce (shallow, 3-conv)'}")
print(f"Augmentation      : {'ENABLED (frame + flow)' if USE_AUGMENTATION else 'DISABLED'}")
print(f"Checkpoint file   : {CHECKPOINT_NAME}")
print("=" * 60)

# ============================================================
# REPRODUCIBILITY
# ============================================================
torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDEVICE : {device}")

USE_AMP = False

ABLATION CONFIGURATION
Config name       : motioncnn_no_aug
Flow branch       : MotionCNN (deep, SE attention)
Augmentation      : DISABLED
Checkpoint file   : best_crowd_model_motioncnn_no_aug.pth

DEVICE : cuda


C:\Users\RAJESH\csrnet-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# CELL 2 : DATASET (augmentation gated by USE_AUGMENTATION)
# ============================================================

WARMUP_FRAMES   = 5
DENSITY_OUT_RES = 28
SCALE_FACTOR    = (512 * 512) / (DENSITY_OUT_RES * DENSITY_OUT_RES)


def augment_frames(frames_np, aug_prob=0.5):
    """
    Random photometric augmentation applied BEFORE Farneback,
    so the resulting flow reflects realistic degraded motion
    estimation rather than clean flow paired with noisy frames.

    Only called when USE_AUGMENTATION is True for this run.
    """
    if np.random.rand() > aug_prob:
        return frames_np

    mode = np.random.choice([
        "gaussian", "blur", "brightness",
        "salt_pepper", "fog", "frame_dropout", "none"
    ], p=[0.18, 0.14, 0.14, 0.18, 0.14, 0.07, 0.15])

    if mode == "none":
        return frames_np

    augmented = []

    if mode == "gaussian":
        sigma = np.random.uniform(5, 20)
        for f in frames_np:
            noise = np.random.normal(0, sigma, f.shape).astype(np.float32)
            augmented.append(np.clip(f.astype(np.float32) + noise, 0, 255).astype(np.uint8))

    elif mode == "blur":
        k = np.random.choice([3, 5, 7, 9])
        for f in frames_np:
            augmented.append(cv2.GaussianBlur(f, (k, k), 0))

    elif mode == "brightness":
        factor = np.random.uniform(0.5, 1.6)
        for f in frames_np:
            augmented.append(np.clip(f.astype(np.float32) * factor, 0, 255).astype(np.uint8))

    elif mode == "salt_pepper":
        prob = np.random.uniform(0.005, 0.05)
        for f in frames_np:
            aug = f.copy()
            mask = np.random.rand(*f.shape)
            aug[mask < prob / 2] = 0
            aug[(mask >= prob / 2) & (mask < prob)] = 255
            augmented.append(aug)

    elif mode == "fog":
        alpha = np.random.uniform(0.15, 0.50)
        for f in frames_np:
            augmented.append(np.clip(
                (1 - alpha) * f.astype(np.float32) + alpha * 128, 0, 255
            ).astype(np.uint8))

    elif mode == "frame_dropout":
        augmented = [f.copy() for f in frames_np]
        n_drop = np.random.choice([1, 2], p=[0.8, 0.2])
        T = len(augmented)
        drop_indices = np.random.choice(np.arange(T), size=n_drop, replace=False)
        for idx in drop_indices:
            if np.random.rand() < 0.5:
                augmented[idx][:] = 0
            else:
                if idx > 0:
                    augmented[idx] = augmented[idx - 1].copy()

    return augmented


def augment_flow(flow_tensor):
    """
    Flow-space augmentation: magnitude scale, horizontal flip,
    mild additive noise. Only called when USE_AUGMENTATION=True.
    """
    scale = np.random.uniform(0.8, 1.2)
    flow_tensor = flow_tensor * scale

    if np.random.rand() < 0.5:
        flow_tensor = torch.flip(flow_tensor, dims=[3])
        flow_tensor[:, 0, :, :] = -flow_tensor[:, 0, :, :]

    if np.random.rand() < 0.1:
        noise_sigma = np.random.uniform(0.03, 0.06)
    else:
        noise_sigma = np.random.uniform(0.005, 0.02)
    flow_tensor = flow_tensor + torch.randn_like(flow_tensor) * noise_sigma

    return torch.clamp(flow_tensor, -1.0, 1.0)


class CrowdDataset(Dataset):

    def __init__(self, base_dir, seq_length=15, stride=4,
                 training=False, aug_prob=0.3):
        self.seq_length = seq_length
        self.training   = training
        self.aug_prob   = aug_prob
        # Augmentation is only ever applied if BOTH the dataset
        # is in training mode AND the global ablation flag enables it
        self.use_aug    = training and USE_AUGMENTATION
        self.windows    = []
        self.classes    = {"0_normal": 0, "1_bottleneck": 1, "2_panic": 2}

        print("=" * 60)
        print(f"LOADING DATASET : {base_dir}")
        print(f"seq_length      : {seq_length}")
        print(f"stride          : {stride}")
        print(f"training mode   : {training}")
        print(f"augmentation    : {'ON' if self.use_aug else 'OFF'} (ablation={ABLATION_CONFIG})")
        print("=" * 60)

        for c_name, c_idx in self.classes.items():
            class_dir = os.path.join(base_dir, c_name)
            if not os.path.exists(class_dir):
                continue
            for seq_folder in sorted(os.listdir(class_dir)):
                seq_path = os.path.join(class_dir, seq_folder)
                if not os.path.isdir(seq_path):
                    continue
                frame_files = sorted(glob.glob(os.path.join(seq_path, "frame_*.png")))
                num_frames = len(frame_files)
                if num_frames < seq_length + 1:
                    continue
                for start_idx in range(WARMUP_FRAMES, num_frames - seq_length, stride):
                    self.windows.append({
                        "seq_path": seq_path, "label": c_idx, "start_idx": start_idx
                    })

        print(f"TOTAL WINDOWS : {len(self.windows)}")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):

        w        = self.windows[idx]
        seq_path = w["seq_path"]
        label    = w["label"]
        start    = w["start_idx"]

        raw_uint8 = []
        for t in range(start, start + self.seq_length + 1):
            img_path = os.path.join(seq_path, f"frame_{t:03d}.png")
            img = Image.open(img_path).convert("L").resize((224, 224))
            raw_uint8.append(np.array(img))

        if self.use_aug:
            raw_uint8 = augment_frames(raw_uint8, aug_prob=self.aug_prob)

        flows = []
        for t in range(self.seq_length):
            flow = cv2.calcOpticalFlowFarneback(
                raw_uint8[t], raw_uint8[t + 1], None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            flow = np.clip(flow / 8.0, -1.0, 1.0)

            # Median pre-filter for S&P spike suppression — only
            # active during augmented training, since this is part
            # of the augmentation pipeline being ablated
            if self.use_aug and np.random.rand() < 0.3:
                flow_x = cv2.medianBlur(flow[:, :, 0], 3)
                flow_y = cv2.medianBlur(flow[:, :, 1], 3)
                flow   = np.stack([flow_x, flow_y], axis=2)

            flows.append(torch.tensor(flow, dtype=torch.float32).permute(2, 0, 1))

        frames = [
            torch.tensor(raw_uint8[t].astype(np.float32) / 255.0,
                        dtype=torch.float32).unsqueeze(0)
            for t in range(self.seq_length)
        ]

        frame_tensor = torch.stack(frames)
        flow_tensor  = torch.stack(flows)

        if self.use_aug:
            flow_tensor = augment_flow(flow_tensor)

        target_idx   = start + self.seq_length - 1
        density_path = os.path.join(seq_path, f"density_{target_idx:03d}.npy")
        density_512  = np.load(density_path).astype(np.float32)
        density_28   = cv2.resize(density_512, (DENSITY_OUT_RES, DENSITY_OUT_RES),
                                  interpolation=cv2.INTER_AREA)
        density_28   = density_28 * SCALE_FACTOR
        density_tensor = torch.tensor(density_28, dtype=torch.float32).unsqueeze(0)

        if idx == 0:
            print(f"\nFLOW VERIFICATION (idx=0)\n"
                  f"  flow std    : {flow_tensor.std():.4f}\n"
                  f"  density sum : {density_tensor.sum():.1f} agents\n")

        return frame_tensor, flow_tensor, torch.tensor(label, dtype=torch.long), density_tensor

In [3]:
# ============================================================
# CELL 3 : MODEL (flow branch gated by USE_MOTIONCNN)
# ============================================================

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(4 * hidden_dim)

    def forward(self, x, states):
        h_cur, c_cur = states
        combined = torch.cat([x, h_cur], dim=1)
        gates    = self.bn(self.conv(combined))
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


# ============================================================
# OPTION A: SHALLOW STRIDED FLOW BLOCK (the original "baseline")
# ============================================================
# 3 stride-2 convolutions, ~4K parameters, output 32 channels.
# This was the architecture BEFORE MotionCNN was introduced.

class StridedFlowReduce(nn.Module):
    """
    Input : (B, 2, 224, 224)
    Output: (B, 32, 7, 7)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d((7, 7))
        )
        self.out_channels = 32

    def forward(self, x):
        return self.net(x)


# ============================================================
# OPTION B: MOTIONCNN — deep flow branch with SE attention
# ============================================================

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.pool(x).view(B, C)
        w = self.fc(w).view(B, C, 1, 1)
        return x * w


class MotionResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        mid_ch = out_ch * 2
        self.pw1 = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch), nn.Hardswish()
        )
        self.dw = nn.Sequential(
            nn.Conv2d(mid_ch, mid_ch, 3, stride=stride, padding=1,
                      groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch), nn.Hardswish()
        )
        self.se = SEBlock(mid_ch)
        self.pw2 = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.use_residual = (stride == 1 and in_ch == out_ch)

    def forward(self, x):
        out = self.pw1(x)
        out = self.dw(out)
        out = self.se(out)
        out = self.pw2(out)
        if self.use_residual:
            out = out + x
        return out


class MotionCNN(nn.Module):
    """
    Input : (B, 2, 224, 224)
    Output: (B, 64, 7, 7)
    """
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.Hardswish()
        )
        self.blocks = nn.Sequential(
            MotionResBlock(16, 24, stride=2),
            MotionResBlock(24, 32, stride=2),
            MotionResBlock(32, 48, stride=1),
            MotionResBlock(48, 64, stride=2),
            MotionResBlock(64, 64, stride=1),
        )
        self.final = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.Hardswish(),
            nn.AdaptiveAvgPool2d((7, 7))
        )
        self.drop = nn.Dropout2d(0.15)
        self.out_channels = 64

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.final(x)
        x = self.drop(x)
        return x


# ============================================================
# CROWDNET — flow branch selected by USE_MOTIONCNN flag
# ============================================================

class CrowdNet(nn.Module):

    def __init__(self):
        super().__init__()

        HIDDEN_DIM      = 128
        self.hidden_dim = HIDDEN_DIM

        mobilenet = models.mobilenet_v3_small(weights=None)
        if os.path.exists("mobilenet_v3_small-047dcff4.pth"):
            print("LOADING LOCAL MOBILENET WEIGHTS")
            state_dict = torch.load("mobilenet_v3_small-047dcff4.pth", weights_only=True)
            mobilenet.load_state_dict(state_dict)

        features = mobilenet.features
        self.backbone_stage1 = features[:3]
        self.backbone_stage2 = features[3:]
        for param in self.backbone_stage1.parameters():
            param.requires_grad = False

        self.density_head = nn.Sequential(
            nn.Conv2d(24, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )

        self.spatial_reduce = nn.Sequential(
            nn.Conv2d(576, 64, kernel_size=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout2d(0.2)
        )

        # ── FLOW BRANCH SELECTION — the key ablation switch ──────
        if USE_MOTIONCNN:
            self.flow_branch = MotionCNN()
            flow_out_channels = self.flow_branch.out_channels   # 64
        else:
            self.flow_branch = StridedFlowReduce()
            flow_out_channels = self.flow_branch.out_channels   # 32

        # ConvLSTM input dimension adapts automatically to whichever
        # flow branch is active: 64+64=128 with MotionCNN,
        # or 64+32=96 with the strided baseline block
        lstm_input_dim = 64 + flow_out_channels

        self.convlstm = ConvLSTMCell(input_dim=lstm_input_dim, hidden_dim=HIDDEN_DIM)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, 3)
        )

    def forward(self, frames, flows):
        B, T, _, H, W = frames.shape
        h, c = None, None
        last_stage1 = None

        for t in range(T):
            frame = frames[:, t]
            flow  = flows[:, t]
            frame_rgb = frame.repeat(1, 3, 1, 1)

            stage1 = self.backbone_stage1(frame_rgb)
            stage2 = self.backbone_stage2(stage1)
            last_stage1 = stage1

            spatial_small = self.spatial_reduce(stage2)
            flow_features = self.flow_branch(flow)   # MotionCNN or StridedFlowReduce

            fused = torch.cat([spatial_small, flow_features], dim=1)

            if h is None:
                _, _, FH, FW = fused.shape
                h = torch.zeros(B, self.hidden_dim, FH, FW, device=fused.device)
                c = torch.zeros_like(h)

            h, c = self.convlstm(fused, (h, c))

        density_map = self.density_head(last_stage1)
        pooled = self.pool(h).view(B, -1)
        logits = self.classifier(pooled)
        return logits, density_map


# ── Instantiate ───────────────────────────────────────────────
model = CrowdNet().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print("=" * 60)
print(f"MODEL INITIALIZED — ABLATION: {ABLATION_CONFIG}")
print(f"Flow branch       : {type(model.flow_branch).__name__}")
print(f"Flow out channels : {model.flow_branch.out_channels}")
print(f"ConvLSTM input dim: {model.convlstm.conv.in_channels - model.hidden_dim}")
print(f"Trainable params  : {trainable:,}")
print(f"Total params      : {total:,}")
print("=" * 60)

with torch.no_grad():
    df  = torch.zeros(1, 15, 1, 224, 224).to(device)
    dfl = torch.zeros(1, 15, 2, 224, 224).to(device)
    dl, dd = model(df, dfl)
    print(f"Logits shape  : {dl.shape}")
    print(f"Density shape : {dd.shape}")

LOADING LOCAL MOBILENET WEIGHTS
MODEL INITIALIZED — ABLATION: motioncnn_no_aug
Flow branch       : MotionCNN
Flow out channels : 64
ConvLSTM input dim: 128
Trainable params  : 2,314,488
Total params      : 2,319,560
Logits shape  : torch.Size([1, 3])
Density shape : torch.Size([1, 1, 28, 28])


In [ ]:
print(model.spatial_reduce)

In [4]:
#cell 4
# ============================================================
# CELL 4 : LOSSES + OPTIMIZER + SCHEDULER
# ============================================================

classification_criterion = nn.CrossEntropyLoss()

density_criterion = nn.SmoothL1Loss(beta=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=5e-4       # increased from 1e-4 to fight overfitting
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,             # reduced from 5 — tighter for 40 epochs
    min_lr=1e-6
)

print("=" * 60)
print("LOSSES + OPTIMIZER INITIALIZED")
print(f"LR           : {optimizer.param_groups[0]['lr']}")
print(f"Weight decay : {optimizer.param_groups[0]['weight_decay']}")
print("=" * 60)

LOSSES + OPTIMIZER INITIALIZED
LR           : 0.0003
Weight decay : 0.0005


In [5]:
# ============================================================
# CELL 5 : TRAINING + VALIDATION LOOPS
# ============================================================

def train_one_epoch(model, loader, optimizer, epoch, total_epochs):
    """
    Curriculum augmentation schedule. Note: if USE_AUGMENTATION
    is False for this ablation run, self.use_aug on the dataset
    was already permanently set to False in Cell 2's __init__,
    so changing aug_prob here has NO EFFECT for the no-aug runs.
    This is intentional — it keeps Cell 5 identical across all
    four ablation configs without needing a separate version.
    """
    if epoch <= 5:
        aug_prob = 0.3
    elif epoch <= 15:
        aug_prob = 0.5
    else:
        aug_prob = 0.7

    loader.dataset.aug_prob = aug_prob

    model.train()
    total_loss = total_cls_loss = total_den_loss = 0.0
    correct = total = 0

    loop = tqdm(enumerate(loader), total=len(loader), leave=True)

    for batch_idx, (frames, flows, labels, density_targets) in loop:

        frames          = frames.to(device)
        flows           = flows.to(device)
        labels          = labels.to(device)
        density_targets = density_targets.to(device)

        optimizer.zero_grad()
        logits, density_preds = model(frames, flows)
        density_preds = torch.clamp(density_preds, min=0.0)

        cls_loss = classification_criterion(logits, labels)
        den_loss = density_criterion(density_preds, density_targets)
        loss     = cls_loss + 0.001 * den_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss     += loss.item()
        total_cls_loss += cls_loss.item()
        total_den_loss += den_loss.item()

        preds    = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        acc      = 100.0 * correct / total

        loop.set_description(
            f"TRAIN [{ABLATION_CONFIG}] ep{epoch} aug={aug_prob:.1f} | "
            f"LOSS {total_loss/(batch_idx+1):.4f} | "
            f"CLS {total_cls_loss/(batch_idx+1):.4f} | "
            f"ACC {acc:.2f}%"
        )

    return (total_loss/len(loader), total_cls_loss/len(loader),
            total_den_loss/len(loader), correct/total)


def validate_one_epoch(model, loader):
    model.eval()
    total_loss = total_cls_loss = total_den_loss = 0.0
    correct = total = 0

    with torch.no_grad():
        loop = tqdm(enumerate(loader), total=len(loader), leave=True)
        for batch_idx, (frames, flows, labels, density_targets) in loop:

            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)
            density_preds = torch.clamp(density_preds, min=0.0)

            cls_loss = classification_criterion(logits, labels)
            den_loss = density_criterion(density_preds, density_targets)
            loss     = cls_loss + 0.001 * den_loss

            total_loss     += loss.item()
            total_cls_loss += cls_loss.item()
            total_den_loss += den_loss.item()

            preds    = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            acc      = 100.0 * correct / total

            loop.set_description(
                f"VAL [{ABLATION_CONFIG}] | "
                f"LOSS {total_loss/(batch_idx+1):.4f} | "
                f"CLS {total_cls_loss/(batch_idx+1):.4f} | "
                f"ACC {acc:.2f}%"
            )

    return (total_loss/len(loader), total_cls_loss/len(loader),
            total_den_loss/len(loader), correct/total)

In [6]:
# ============================================================
# CELL 6 : TRAINING DRIVER
# ============================================================

train_dataset = CrowdDataset(
    "split_dataset_v7/train", seq_length=15, stride=4,
    training=True, aug_prob=0.3
)
val_dataset = CrowdDataset(
    "split_dataset_v7/val", seq_length=15, stride=4, training=False
)
test_dataset = CrowdDataset(
    "split_dataset_v7/test", seq_length=15, stride=4, training=False
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"Train windows : {len(train_dataset)}")
print(f"Val windows   : {len(val_dataset)}")
print(f"Test windows  : {len(test_dataset)}")

sample_frames, sample_flows, sample_labels, sample_density = next(iter(train_loader))
print(f"\nFrames shape  : {sample_frames.shape}")
print(f"Flows shape   : {sample_flows.shape}")
print(f"Flow std      : {sample_flows.std():.4f}")

print("\n=== FULL LABEL DISTRIBUTION ===")
for split_name, dataset in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    counts = {0: 0, 1: 0, 2: 0}
    for w in dataset.windows:
        counts[w["label"]] += 1
    print(f"{split_name:6s} | {counts} | total: {sum(counts.values())}")

train_std = sample_flows.std().item()
assert train_std > 0.03, f"Flow std {train_std:.4f} too low — check Cell 2."

print(f"\nSANITY CHECKS PASSED — starting training for [{ABLATION_CONFIG}]\n")

EPOCHS = 50
best_val_loss = 1e9
best_val_acc  = 0.0
patience = 10
patience_counter = 0

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"[{ABLATION_CONFIG}] EPOCH {epoch+1}/{EPOCHS}")
    print("=" * 60)

    train_loss, train_cls, train_den, train_acc = train_one_epoch(
        model, train_loader, optimizer, epoch+1, EPOCHS
    )
    val_loss, val_cls, val_den, val_acc = validate_one_epoch(model, val_loader)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"\nTRAIN  | loss {train_loss:.4f} | cls {train_cls:.4f} | den {train_den:.4f} | acc {train_acc*100:.2f}%")
    print(f"VAL    | loss {val_loss:.4f} | cls {val_cls:.4f} | den {val_den:.4f} | acc {val_acc*100:.2f}%")
    print(f"LR     : {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_NAME)
        print(f"MODEL IMPROVED → SAVED to {CHECKPOINT_NAME} (val_acc={val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"NO IMPROVEMENT ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("\nEARLY STOPPING TRIGGERED")
        break

print(f"\n[{ABLATION_CONFIG}] Best val loss : {best_val_loss:.4f}")
print(f"[{ABLATION_CONFIG}] Best val acc  : {best_val_acc*100:.2f}%")
print("\nTRAINING COMPLETE")

LOADING DATASET : split_dataset_v7/train
seq_length      : 15
stride          : 4
training mode   : True
augmentation    : ON (ablation=motioncnn_aug)
TOTAL WINDOWS : 1260
LOADING DATASET : split_dataset_v7/val
seq_length      : 15
stride          : 4
training mode   : False
augmentation    : OFF (ablation=motioncnn_aug)
TOTAL WINDOWS : 270
LOADING DATASET : split_dataset_v7/test
seq_length      : 15
stride          : 4
training mode   : False
augmentation    : OFF (ablation=motioncnn_aug)
TOTAL WINDOWS : 270
Train windows : 1260
Val windows   : 270
Test windows  : 270

Frames shape  : torch.Size([4, 15, 1, 224, 224])
Flows shape   : torch.Size([4, 15, 2, 224, 224])
Flow std      : 0.1108

=== FULL LABEL DISTRIBUTION ===
train  | {0: 420, 1: 420, 2: 420} | total: 1260
val    | {0: 90, 1: 90, 2: 90} | total: 270
test   | {0: 90, 1: 90, 2: 90} | total: 270

SANITY CHECKS PASSED — starting training for [motioncnn_aug]


[motioncnn_aug] EPOCH 1/50


TRAIN [motioncnn_aug] ep1 aug=0.3 | LOSS 0.8268 | CLS 0.8268 | ACC 58.41%:  68%|███▍ | 214/315 [08:33<03:59,  2.37s/it]


FLOW VERIFICATION (idx=0)
  flow std    : 0.0899
  density sum : 151.0 agents



TRAIN [motioncnn_aug] ep1 aug=0.3 | LOSS 0.7357 | CLS 0.7357 | ACC 64.60%: 100%|█████| 315/315 [12:36<00:00,  2.40s/it]
  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1000
  density sum : 138.0 agents



...


FLOW VERIFICATION (idx=0)
  flow std    : 0.0916
  density sum : 151.0 agents



TRAIN [motioncnn_aug] ep18 aug=0.7 | LOSS 0.0085 | CLS 0.0085 | ACC 99.68%: 100%|████| 315/315 [12:38<00:00,  2.41s/it]
  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1000
  density sum : 138.0 agents



VAL [motioncnn_aug] | LOSS 0.0993 | CLS 0.0993 | ACC 98.15%: 100%|█████████████████████| 68/68 [01:47<00:00,  1.58s/it]


TRAIN  | loss 0.0085 | cls 0.0085 | den 0.0072 | acc 99.68%
VAL    | loss 0.0993 | cls 0.0993 | den 0.0058 | acc 98.15%
LR     : 0.000150
NO IMPROVEMENT (10/10)

EARLY STOPPING TRIGGERED

[motioncnn_aug] Best val loss : 0.0387
[motioncnn_aug] Best val acc  : 98.89%

TRAINING COMPLETE


In [6]:
# ============================================================
# CELL 7 : TESTING + STANDARD ROBUSTNESS
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
test_dataset = CrowdDataset(
    "split_dataset_v7/test", seq_length=15, stride=4, training=False
)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)
def corrupt_frames(frames, flows, mode="gaussian"):
    frames = frames.clone()
    flows  = flows.clone()

    if mode == "gaussian":
        frames = frames + torch.randn_like(frames) * 0.08
        flows  = flows  + torch.randn_like(flows)  * 0.02
    elif mode == "blur":
        B, T, C, H, W = frames.shape
        blurred = []
        for b in range(B):
            temp = []
            for t in range(T):
                x = frames[b, t].cpu().numpy()[0]
                x = cv2.GaussianBlur(x, (9, 9), 0)
                temp.append(torch.tensor(x).unsqueeze(0))
            blurred.append(torch.stack(temp))
        frames = torch.stack(blurred)
        flows  = flows * 0.85

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)
    return frames, flows


def evaluate_model(model, loader, corruption=None):
    model.eval()
    preds_all, labels_all = [], []
    mae_total, n_batches = 0.0, 0

    with torch.no_grad():
        loop = tqdm(loader)
        for frames, flows, labels, density_targets in loop:
            if corruption is not None:
                frames, flows = corrupt_frames(frames, flows, corruption)

            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)
            density_preds = torch.clamp(density_preds, min=0.0)

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pred_counts = density_preds.sum(dim=[1,2,3]).cpu().numpy()
            true_counts = density_targets.sum(dim=[1,2,3]).cpu().numpy()
            mae_total  += np.mean(np.abs(pred_counts - true_counts))
            n_batches  += 1

    acc = accuracy_score(labels_all, preds_all)
    mae = mae_total / n_batches

    print("\n" + "=" * 60)
    print(f"[{ABLATION_CONFIG}] " + ("CLEAN EVALUATION" if corruption is None else f"CORRUPTION : {corruption}"))
    print("=" * 60)
    print(f"\nACCURACY  : {acc*100:.2f}%")
    print(f"COUNT MAE : {mae:.2f} agents")
    print("\nCLASSIFICATION REPORT\n")
    print(classification_report(labels_all, preds_all,
                                target_names=["Normal", "Bottleneck", "Panic"],
                                digits=4, zero_division=0))
    print("\nCONFUSION MATRIX\n")
    print(confusion_matrix(labels_all, preds_all))

    return acc * 100, mae   # RETURN values so they can be reused, never hardcoded downstream


# ── Load and evaluate ────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT_NAME, weights_only=True))
print(f"BEST MODEL LOADED for [{ABLATION_CONFIG}]")

clean_acc, clean_mae = evaluate_model(model, test_loader, corruption=None)
gauss_acc, gauss_mae = evaluate_model(model, test_loader, corruption="gaussian")
blur_acc,  blur_mae  = evaluate_model(model, test_loader, corruption="blur")

# Store this run's results for the final ablation comparison table
ablation_results = {
    "config": ABLATION_CONFIG,
    "clean_acc": clean_acc, "clean_mae": clean_mae,
    "gauss_acc": gauss_acc, "gauss_mae": gauss_mae,
    "blur_acc":  blur_acc,  "blur_mae":  blur_mae,
}
print(f"\nablation_results dict populated for [{ABLATION_CONFIG}]:")
print(ablation_results)

LOADING DATASET : split_dataset_v7/test
seq_length      : 15
stride          : 4
training mode   : False
augmentation    : OFF (ablation=motioncnn_no_aug)
TOTAL WINDOWS : 270
BEST MODEL LOADED for [motioncnn_no_aug]


  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents



100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [01:46<00:00,  1.57s/it]



[motioncnn_no_aug] CLEAN EVALUATION

ACCURACY  : 97.04%
COUNT MAE : 1.90 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.9881    0.9222    0.9540        90
  Bottleneck     0.9271    0.9889    0.9570        90
       Panic     1.0000    1.0000    1.0000        90

    accuracy                         0.9704       270
   macro avg     0.9717    0.9704    0.9703       270
weighted avg     0.9717    0.9704    0.9703       270


CONFUSION MATRIX

[[83  7  0]
 [ 1 89  0]
 [ 0  0 90]]


...


[motioncnn_no_aug] CORRUPTION : gaussian

ACCURACY  : 92.22%
COUNT MAE : 85.21 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.8350    0.9556    0.8912        90
  Bottleneck     1.0000    0.8111    0.8957        90
       Panic     0.9574    1.0000    0.9783        90

    accuracy                         0.9222       270
   macro avg     0.9308    0.9222    0.9217       270
weighted avg     0.9308    0.9222    0.9217       270


CONFUSION MATRIX

[[86  0  4]
 [17 73  0]
 [ 0  0 90]]


  0%|                                                                                           | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents



100%|██████████████████████████████████████████████████████████████████████████████████| 68/68 [01:46<00:00,  1.56s/it]


[motioncnn_no_aug] CORRUPTION : blur

ACCURACY  : 97.78%
COUNT MAE : 22.17 agents

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Normal     0.9565    0.9778    0.9670        90
  Bottleneck     0.9773    0.9556    0.9663        90
       Panic     1.0000    1.0000    1.0000        90

    accuracy                         0.9778       270
   macro avg     0.9779    0.9778    0.9778       270
weighted avg     0.9779    0.9778    0.9778       270


CONFUSION MATRIX

[[88  2  0]
 [ 4 86  0]
 [ 0  0 90]]

ablation_results dict populated for [motioncnn_no_aug]:
{'config': 'motioncnn_no_aug', 'clean_acc': 97.03703703703704, 'clean_mae': 1.9044097171110266, 'gauss_acc': 92.22222222222223, 'gauss_mae': 85.2134708516738, 'blur_acc': 97.77777777777777, 'blur_mae': 22.17021888845107}


In [7]:
# ============================================================
# CELL 8 : TEMPORAL SANITY TESTS
# ============================================================

from sklearn.metrics import accuracy_score

def temporal_shuffle(frames, flows):
    B, T, C, H, W = frames.shape
    sf, sfl = [], []
    for b in range(B):
        perm = torch.randperm(T)
        sf.append(frames[b, perm]); sfl.append(flows[b, perm])
    return torch.stack(sf), torch.stack(sfl)

def duplicate_single_frame(frames, flows):
    B, T, C, H, W = frames.shape
    df, dfl = [], []
    for b in range(B):
        idx = T // 2
        frame = frames[b, idx]
        df.append(frame.unsqueeze(0).repeat(T, 1, 1, 1))
        dfl.append(torch.zeros_like(flows[b]))
    return torch.stack(df), torch.stack(dfl)

def zero_flow(frames, flows):
    return frames, torch.zeros_like(flows)

def random_flow(frames, flows):
    rf = torch.clamp(torch.randn_like(flows), -1.0, 1.0)
    return frames, rf

def reverse_temporal(frames, flows):
    return torch.flip(frames, dims=[1]), torch.flip(flows, dims=[1])


def evaluate_sanity_mode(model, loader, mode_name, transform_fn=None):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        loop = tqdm(loader, desc=f"[{ABLATION_CONFIG}] TEST : {mode_name}")
        for frames, flows, labels, density_targets in loop:
            if transform_fn is not None:
                frames, flows = transform_fn(frames, flows)
            frames = frames.to(device); flows = flows.to(device); labels = labels.to(device)
            logits, _ = model(frames, flows)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    print(f"\nSANITY TEST [{ABLATION_CONFIG}] : {mode_name} → Accuracy : {acc*100:.2f}%")
    return acc


def run_temporal_sanity_suite(model, test_loader):
    print(f"\n{'='*70}\nTEMPORAL SANITY SUITE — [{ABLATION_CONFIG}]\n{'='*70}")

    results = {}
    results["normal"]      = evaluate_sanity_mode(model, test_loader, "NORMAL", None)
    results["shuffled"]    = evaluate_sanity_mode(model, test_loader, "TEMPORAL SHUFFLE", temporal_shuffle)
    results["reversed"]    = evaluate_sanity_mode(model, test_loader, "TEMPORAL REVERSE", reverse_temporal)
    results["duplicated"]  = evaluate_sanity_mode(model, test_loader, "SINGLE FRAME DUP", duplicate_single_frame)
    results["zero_flow"]   = evaluate_sanity_mode(model, test_loader, "ZERO FLOW", zero_flow)
    results["random_flow"] = evaluate_sanity_mode(model, test_loader, "RANDOM FLOW", random_flow)

    print(f"\n{'='*70}\nSANITY SUMMARY — [{ABLATION_CONFIG}]\n{'='*70}")
    for k, v in results.items():
        print(f"{k:25s} : {v*100:.2f}%")

    shuffle_drop = (results["normal"] - results["shuffled"]) * 100
    reverse_drop = (results["normal"] - results["reversed"]) * 100
    dup_drop     = (results["normal"] - results["duplicated"]) * 100
    zflow_drop   = (results["normal"] - results["zero_flow"]) * 100

    print(f"\nTemporal Shuffle Drop : {shuffle_drop:.2f}%")
    print(f"Temporal Reverse Drop : {reverse_drop:.2f}%")
    print(f"Single Frame Drop     : {dup_drop:.2f}%")
    print(f"Zero Flow Drop        : {zflow_drop:.2f}%")

    # Append to ablation_results dict (continuing from Cell 7)
    ablation_results.update({
        "shuffle_drop": shuffle_drop, "reverse_drop": reverse_drop,
        "dup_drop": dup_drop, "zflow_drop": zflow_drop,
        "sanity_normal": results["normal"] * 100,
    })

    return results

sanity_results = run_temporal_sanity_suite(model, test_loader)


TEMPORAL SANITY SUITE — [motioncnn_no_aug]


[motioncnn_no_aug] TEST : NORMAL:   0%|                                                         | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents



[motioncnn_no_aug] TEST : NORMAL: 100%|████████████████████████████████████████████████| 68/68 [01:43<00:00,  1.52s/it]



SANITY TEST [motioncnn_no_aug] : NORMAL → Accuracy : 97.04%


...


SANITY TEST [motioncnn_no_aug] : ZERO FLOW → Accuracy : 35.19%


[motioncnn_no_aug] TEST : RANDOM FLOW:   0%|                                                    | 0/68 [00:00<?, ?it/s]


FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents



[motioncnn_no_aug] TEST : RANDOM FLOW: 100%|███████████████████████████████████████████| 68/68 [01:49<00:00,  1.61s/it]


SANITY TEST [motioncnn_no_aug] : RANDOM FLOW → Accuracy : 33.33%

SANITY SUMMARY — [motioncnn_no_aug]
normal                    : 97.04%
shuffled                  : 89.26%
reversed                  : 71.11%
duplicated                : 34.07%
zero_flow                 : 35.19%
random_flow               : 33.33%

Temporal Shuffle Drop : 7.78%
Temporal Reverse Drop : 25.93%
Single Frame Drop     : 62.96%
Zero Flow Drop        : 61.85%


In [ ]:
# ============================================================
# Revised CELL 9 : COMPREHENSIVE NOISE SWEEP + VISUALISATION
# ============================================================
#
# Tests each attack type at 6-8 severity levels to produce
# accuracy vs severity curves for the paper.
#
# Attacks covered:
#   1. Gaussian noise (frame sigma)
#   2. Salt and pepper (pixel probability)
#   3. Fog (contrast attenuation alpha)
#   4. Frame dropout (frame loss probability)
#   5. Motion blur (kernel size)
#   6. Brightness (multiplicative factor)
#   7. Combined fog + gaussian
#
# ============================================================

import torch
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from torch.utils.data import DataLoader

# ── Ensure we use the correct stride-5 test loader ────────────
test_dataset_eval = CrowdDataset(
    "split_dataset_v7/test",
    seq_length=15,
    stride=5,
    training=False
)
test_loader_eval = DataLoader(
    test_dataset_eval,
    batch_size=4,
    shuffle=False,
    num_workers=0
)
print(f"Test windows for sweep : {len(test_dataset_eval)}")

# ── Load best model ───────────────────────────────────────────
model.load_state_dict(
    torch.load("best_crowd_model.pth", weights_only=True)
)
model.eval()
print("Best model loaded for sweep.")


# ============================================================
# CORE ATTACK FUNCTION
# ============================================================

def apply_sweep_attack(frames, flows, attack_type, severity):
    """
    Applies a single attack at a given severity level.
    All inputs/outputs normalised: frames [0,1], flows [-1,1].

    severity meaning per attack:
      gaussian    : frame sigma (0.0 to 0.20)
      salt_pepper : pixel probability (0.0 to 0.10)
      fog         : alpha, contrast pull toward 0.5 (0.0 to 0.80)
      frame_drop  : probability of zeroing a frame (0.0 to 0.60)
      blur        : gaussian kernel size (1 to 19, odd only)
      brightness  : multiplicative factor (0.2 to 2.0)
      fog_gauss   : combined alpha=severity, sigma=severity*0.15
    """
    frames = frames.clone().float()
    flows  = flows.clone().float()

    if attack_type == "gaussian":
        sigma  = severity
        frames = frames + torch.randn_like(frames) * sigma
        # Farneback noise transfer: empirically ~20% of frame sigma
        flows  = flows  + torch.randn_like(flows)  * (sigma * 0.20)

    elif attack_type == "salt_pepper":
        prob = severity
        mask = torch.rand_like(frames)
        frames[mask < prob / 2]                    = 0.0
        frames[(mask >= prob/2) & (mask < prob)]   = 1.0
        # Flow spike suppression via median filter
        flows_np = flows.cpu().numpy()
        B, T, C, H, W = flows_np.shape
        for b in range(B):
            for t in range(T):
                flows_np[b, t, 0] = cv2.medianBlur(
                    flows_np[b, t, 0].astype(np.float32), 3
                )
                flows_np[b, t, 1] = cv2.medianBlur(
                    flows_np[b, t, 1].astype(np.float32), 3
                )
        flows = torch.from_numpy(flows_np)
        # Residual flow spikes
        fmask = torch.rand_like(flows)
        flows[fmask < prob / 8]                        = -1.0
        flows[(fmask >= prob/8) & (fmask < prob/4)]    =  1.0

    elif attack_type == "fog":
        alpha  = severity
        frames = (1 - alpha) * frames + alpha * 0.5
        flows  = flows * (1 - alpha * 0.6)

    elif attack_type == "frame_drop":
        prob = severity
        B, T, C, H, W = frames.shape
        for b in range(B):
            for t in range(T):
                if torch.rand(1).item() < prob:
                    frames[b, t] = 0.0
                    flows[b, t]  = 0.0

    elif attack_type == "blur":
        k = int(severity)
        if k % 2 == 0:
            k += 1   # kernel must be odd
        if k < 1:
            k = 1
        if k > 1:
            B, T, C, H, W = frames.shape
            blurred = []
            for b in range(B):
                temp = []
                for t in range(T):
                    x = frames[b, t].cpu().numpy()[0]
                    x = cv2.GaussianBlur(x, (k, k), 0)
                    temp.append(torch.tensor(x).unsqueeze(0))
                blurred.append(torch.stack(temp))
            frames = torch.stack(blurred)
        # Flow attenuation increases with blur strength
        attenuation = 1.0 - (k - 1) / 40.0
        flows = flows * max(attenuation, 0.3)

    elif attack_type == "brightness":
        factor = severity
        frames = frames * factor
        flows  = flows * (0.85 + factor * 0.10)

    elif attack_type == "fog_gauss":
        alpha = severity
        sigma = severity * 0.15
        frames = (1 - alpha) * frames + alpha * 0.5
        frames = frames + torch.randn_like(frames) * sigma
        flows  = flows * (1 - alpha * 0.5) + torch.randn_like(flows) * (sigma * 0.15)

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)
    return frames, flows


# ============================================================
# SINGLE SEVERITY EVALUATION
# ============================================================

def evaluate_at_severity(model, loader, attack_type, severity):
    """Returns (overall_acc, normal_acc, bottleneck_acc, panic_acc)."""
    model.eval()
    preds_all  = []
    labels_all = []

    with torch.no_grad():
        for frames, flows, labels, _ in loader:
            frames, flows = apply_sweep_attack(
                frames, flows, attack_type, severity
            )
            frames = frames.to(device)
            flows  = flows.to(device)
            labels = labels.to(device)

            logits, _ = model(frames, flows)
            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    preds_np  = np.array(preds_all)
    labels_np = np.array(labels_all)
    overall   = 100 * accuracy_score(labels_np, preds_np)

    # Per-class accuracy
    per_class = []
    for c in range(3):
        mask = labels_np == c
        if mask.sum() > 0:
            per_class.append(
                100 * (preds_np[mask] == labels_np[mask]).mean()
            )
        else:
            per_class.append(0.0)

    return overall, per_class[0], per_class[1], per_class[2]


# ============================================================
# SWEEP DEFINITIONS
# ============================================================

attacks = {
    "Gaussian Noise\n(frame sigma)": {
        "type":       "gaussian",
        "severities": [0.00, 0.02, 0.04, 0.06, 0.08,
                       0.10, 0.12, 0.15, 0.18, 0.20],
        "x_label":    "Frame sigma",
        "x_format":   "{:.2f}",
    },
    "Salt & Pepper\n(pixel probability)": {
        "type":       "salt_pepper",
        "severities": [0.000, 0.005, 0.010, 0.020,
                       0.030, 0.050, 0.070, 0.100],
        "x_label":    "Pixel probability",
        "x_format":   "{:.3f}",
    },
    "Fog / Haze\n(contrast alpha)": {
        "type":       "fog",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60, 0.70, 0.80],
        "x_label":    "Alpha (0=clear, 1=total grey)",
        "x_format":   "{:.2f}",
    },
    "Frame Dropout\n(loss probability)": {
        "type":       "frame_drop",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60],
        "x_label":    "Frame loss probability",
        "x_format":   "{:.2f}",
    },
    "Motion Blur\n(kernel size)": {
        "type":       "blur",
        "severities": [1, 3, 5, 7, 9, 11, 13, 15, 17, 19],
        "x_label":    "Gaussian kernel size (px)",
        "x_format":   "{:.0f}",
    },
    "Brightness Change\n(multiplicative factor)": {
        "type":       "brightness",
        "severities": [0.25, 0.40, 0.55, 0.70, 0.85,
                       1.00, 1.20, 1.40, 1.60, 1.80],
        "x_label":    "Scale factor (1.0 = clean)",
        "x_format":   "{:.2f}",
    },
    "Combined Fog+Gaussian\n(joint severity)": {
        "type":       "fog_gauss",
        "severities": [0.00, 0.10, 0.20, 0.30,
                       0.40, 0.50, 0.60, 0.70],
        "x_label":    "Joint severity (alpha; sigma=alpha×0.15)",
        "x_format":   "{:.2f}",
    },
}

CLASS_NAMES  = ["Normal", "Bottleneck", "Panic"]
CLASS_COLORS = ["#2196F3", "#FF9800", "#F44336"]


# ============================================================
# RUN ALL SWEEPS
# ============================================================

print("\n" + "="*60)
print("RUNNING COMPREHENSIVE NOISE SWEEPS")
print("This will take approximately 15-25 minutes.")
print("="*60)

results = {}   # attack_name → dict of lists

for attack_name, cfg in attacks.items():
    print(f"\n--- {attack_name.replace(chr(10), ' ')} ---")
    overall_list = []
    normal_list  = []
    bn_list      = []
    panic_list   = []

    for sev in cfg["severities"]:
        label = cfg["x_format"].format(sev)
        overall, n_acc, bn_acc, p_acc = evaluate_at_severity(
            model, test_loader_eval, cfg["type"], sev
        )
        overall_list.append(overall)
        normal_list.append(n_acc)
        bn_list.append(bn_acc)
        panic_list.append(p_acc)
        print(f"  severity={label:>8}  overall={overall:>6.2f}%  "
              f"N={n_acc:>6.2f}%  B={bn_acc:>6.2f}%  P={p_acc:>6.2f}%")

    results[attack_name] = {
        "severities": cfg["severities"],
        "x_label":    cfg["x_label"],
        "x_format":   cfg["x_format"],

        "overall":    overall_list,
        "Normal":     normal_list,
        "Bottleneck": bn_list,
        "Panic":      panic_list,
    }
    
print("\nAll sweeps complete.")



In [ ]:
# ============================================================
# PATCH EXISTING RESULTS DICTIONARY
# ============================================================

for attack_name, cfg in attacks.items():

    if attack_name in results:

        results[attack_name]["x_format"] = cfg["x_format"]

print("results dictionary patched successfully.")

In [ ]:
#Revised cell 9 plots
# ============================================================
# PLOT — 7-panel accuracy vs severity figure
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.patch.set_facecolor("#0d1117")
axes_flat = axes.flatten()

# Hide the 8th panel (2x4 = 8, we have 7 attacks)
axes_flat[7].set_visible(False)

OVERALL_COLOR = "#FFFFFF"
LINE_STYLES   = ["-", "--", "-."]
ALPHA_FILL    = 0.12

for ax_idx, (attack_name, data) in enumerate(results.items()):
    ax = axes_flat[ax_idx]
    ax.set_facecolor("#161b22")

    sevs     = data["severities"]
    overall  = data["overall"]
    normal   = data["Normal"]
    bn       = data["Bottleneck"]
    panic    = data["Panic"]

    # Overall accuracy — thick white line
    ax.plot(sevs, overall, color=OVERALL_COLOR, linewidth=2.5,
            marker="o", markersize=5, label="Overall", zorder=5)

    # Per-class accuracy lines
    for cls_name, cls_data, col, ls in zip(
        CLASS_NAMES,
        [normal, bn, panic],
        CLASS_COLORS,
        LINE_STYLES
    ):
        ax.plot(sevs, cls_data, color=col, linewidth=1.4,
                linestyle=ls, marker="s", markersize=3.5,
                alpha=0.85, label=cls_name)
        # Shaded region between class line and overall
        ax.fill_between(sevs, cls_data, overall,
                        color=col, alpha=ALPHA_FILL)

    # Reference lines
    ax.axhline(y=95, color="#444444", linestyle=":", linewidth=0.8)
    ax.axhline(y=80, color="#333333", linestyle=":", linewidth=0.8)
    ax.axhline(y=33.33, color="#222222", linestyle="--",
               linewidth=0.7, label="Chance (33%)")

    # Shade the operational zone (>90%)
    ax.axhspan(90, 100, alpha=0.04, color="#4CAF50")

    # Annotate clean baseline
    clean_acc = overall[0]
    ax.annotate(
        f"Clean\n{clean_acc:.1f}%",
        xy=(sevs[0], clean_acc),
        xytext=(sevs[0], clean_acc - 12),
        fontsize=7, color="#AAAAAA",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.8),
        ha="center"
    )

    # Find the point where overall drops below 90%
    cliff_sev = None
    for i in range(1, len(overall)):
        if overall[i] < 90.0:
            cliff_sev = sevs[i]
            ax.axvline(x=cliff_sev, color="#FF5722",
                       linestyle="--", linewidth=1.0, alpha=0.7)
            ax.text(cliff_sev, 36,
                    f"<90%\n@{data['x_format'].format(cliff_sev)}",
                    color="#FF5722", fontsize=6.5, ha="center",
                    va="bottom")
            break

    ax.set_xlim(min(sevs), max(sevs))
    ax.set_ylim(25, 105)
    ax.set_xlabel(data["x_label"], color="#CCCCCC", fontsize=8)
    ax.set_ylabel("Accuracy (%)", color="#CCCCCC", fontsize=8)
    ax.set_title(attack_name, color="white", fontsize=10,
                 fontweight="bold", pad=8)
    ax.tick_params(colors="#AAAAAA", labelsize=7)
    ax.spines["bottom"].set_color("#444444")
    ax.spines["left"].set_color("#444444")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#222222", linewidth=0.5)

# Shared legend (bottom of figure)
legend_elements = [
    plt.Line2D([0], [0], color=OVERALL_COLOR, linewidth=2.5,
               marker="o", markersize=5, label="Overall"),
    plt.Line2D([0], [0], color=CLASS_COLORS[0], linewidth=1.4,
               linestyle="-", marker="s", markersize=3.5, label="Normal"),
    plt.Line2D([0], [0], color=CLASS_COLORS[1], linewidth=1.4,
               linestyle="--", marker="s", markersize=3.5, label="Bottleneck"),
    plt.Line2D([0], [0], color=CLASS_COLORS[2], linewidth=1.4,
               linestyle="-.", marker="s", markersize=3.5, label="Panic"),
    plt.Line2D([0], [0], color="#FF5722", linewidth=1.0,
               linestyle="--", label="90% threshold crossing"),
    mpatches.Patch(facecolor="#4CAF50", alpha=0.15,
                   label="Operational zone (>90%)"),
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=6,
    fontsize=9,
    frameon=True,
    facecolor="#1c2128",
    edgecolor="#444444",
    labelcolor="white",
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle(
    "CrowdNet Robustness — Accuracy vs Noise Attack Severity\n"
    "V7 Generator · MotionCNN · ConvLSTM(128) · seq_len=15 · stride=5",
    color="white", fontsize=13, fontweight="bold", y=1.01
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(
    "crowdnet_robustness_sweep.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="#0d1117"
)
plt.close()
print("Saved: crowdnet_robustness_sweep.png")


# ============================================================
# SUMMARY TABLE — operational thresholds
# ============================================================

print("\n" + "="*70)
print("OPERATIONAL NOISE BUDGET SUMMARY")
print("(severity at which overall accuracy first drops below 90%)")
print("="*70)
print(f"{'Attack':<40}  {'Threshold':>12}  {'Clean acc':>10}")
print("-"*70)

for attack_name, data in results.items():
    clean_acc  = data["overall"][0]
    threshold  = "Never <90%"
    for i in range(1, len(data["overall"])):
        if data["overall"][i] < 90.0:
            sev_label = attacks[attack_name]["x_format"].format(
                data["severities"][i]
            )
            unit      = attacks[attack_name]["x_label"].split("(")[0].strip()
            threshold = f"{sev_label} {unit}"
            break
    name_clean = attack_name.replace("\n", " ")
    print(f"{name_clean:<40}  {threshold:>12}  {clean_acc:>9.2f}%")

print("="*70)


# ============================================================
# SECOND PLOT — consolidated overview bar chart
# ============================================================

# For each attack, show accuracy at 3 representative severities:
# mild (25th percentile), moderate (50th), severe (75th)

fig2, ax2 = plt.subplots(figsize=(16, 7))
fig2.patch.set_facecolor("#0d1117")
ax2.set_facecolor("#161b22")

attack_names_short = [
    "Gaussian", "Salt+Pepper", "Fog",
    "Frame\nDropout", "Blur", "Brightness", "Fog+Gauss"
]

x         = np.arange(len(attacks))
bar_width  = 0.22
severity_labels = ["Mild", "Moderate", "Severe"]
severity_colors = ["#4CAF50", "#FF9800", "#F44336"]

for s_idx, (s_label, s_color) in enumerate(
    zip(severity_labels, severity_colors)
):
    accs = []
    for attack_name, data in results.items():
        n    = len(data["severities"])
        idx  = [n // 4, n // 2, 3 * n // 4][s_idx]
        idx  = min(idx, n - 1)
        accs.append(data["overall"][idx])

    bars = ax2.bar(
        x + s_idx * bar_width,
        accs,
        bar_width,
        label=s_label,
        color=s_color,
        alpha=0.85,
        edgecolor="#000000",
        linewidth=0.5
    )

    for bar, acc in zip(bars, accs):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{acc:.0f}%",
            ha="center", va="bottom",
            color="white", fontsize=7, fontweight="bold"
        )

ax2.axhline(y=90, color="#FF5722", linestyle="--",
            linewidth=1.2, label="90% operational threshold")
ax2.axhline(y=33.33, color="#555555", linestyle=":",
            linewidth=0.8, label="Chance level (33.3%)")

ax2.set_xticks(x + bar_width)
ax2.set_xticklabels(attack_names_short, color="white", fontsize=9)
ax2.set_ylim(20, 110)
ax2.set_ylabel("Accuracy (%)", color="white", fontsize=10)
ax2.set_title(
    "CrowdNet Robustness Overview — Accuracy at Mild / Moderate / Severe Corruption",
    color="white", fontsize=12, fontweight="bold", pad=12
)
ax2.tick_params(colors="white", labelsize=9)
ax2.spines["bottom"].set_color("#444444")
ax2.spines["left"].set_color("#444444")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(axis="y", color="#222222", linewidth=0.5)
ax2.legend(
    fontsize=9, frameon=True,
    facecolor="#1c2128", edgecolor="#444444",
    labelcolor="white", loc="lower right"
)

plt.tight_layout()
plt.savefig(
    "crowdnet_robustness_overview.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="#0d1117"
)
plt.close()
print("Saved: crowdnet_robustness_overview.png")

In [8]:
# ============================================================
# CELL 9 : EXTENDED ROBUSTNESS ATTACKS
# ============================================================

import cv2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def apply_attack(frames, flows, mode, params):
    frames = frames.clone().float()
    flows  = flows.clone().float()

    if mode == "salt_pepper":
        prob = params["prob"]
        mask = torch.rand_like(frames)
        frames[mask < prob / 2] = 0.0
        frames[(mask >= prob/2) & (mask < prob)] = 1.0
        flows_np = flows.cpu().numpy()
        B, T, C, H, W = flows_np.shape
        for b in range(B):
            for t in range(T):
                flows_np[b, t, 0] = cv2.medianBlur(flows_np[b, t, 0].astype(np.float32), 3)
                flows_np[b, t, 1] = cv2.medianBlur(flows_np[b, t, 1].astype(np.float32), 3)
        flows = torch.from_numpy(flows_np)
        fmask = torch.rand_like(flows)
        flows[fmask < prob/8] = -1.0
        flows[(fmask >= prob/8) & (fmask < prob/4)] = 1.0

    elif mode == "fog":
        alpha = params["intensity"]
        frames = (1 - alpha) * frames + alpha * 0.5
        flows  = flows * (1 - alpha * 0.6)

    elif mode == "frame_dropout":
        prob = params["prob"]
        B, T, C, H, W = frames.shape
        for b in range(B):
            for t in range(T):
                if torch.rand(1).item() < prob:
                    frames[b, t] = 0.0
                    flows[b, t]  = 0.0

    elif mode == "fog_gaussian":
        alpha = params["fog_intensity"]
        sigma = params["gaussian_sigma"]
        frames = (1 - alpha) * frames + alpha * 0.5
        frames = frames + torch.randn_like(frames) * sigma
        flows  = flows * (1 - alpha * 0.5) + torch.randn_like(flows) * 0.015

    elif mode == "heavy_blur":
        k = params["kernel"]
        B, T, C, H, W = frames.shape
        blurred = []
        for b in range(B):
            temp = []
            for t in range(T):
                x = frames[b, t].cpu().numpy()[0]
                x = cv2.GaussianBlur(x, (k, k), 0)
                temp.append(torch.tensor(x).unsqueeze(0))
            blurred.append(torch.stack(temp))
        frames = torch.stack(blurred)
        flows = flows * 0.65

    elif mode == "brightness":
        factor = params["factor"]
        frames = frames * factor
        flows  = flows * (0.9 + factor * 0.1)

    # mode == None: no corruption, returns clean frames/flows unchanged

    frames = torch.clamp(frames, 0.0, 1.0)
    flows  = torch.clamp(flows, -1.0, 1.0)
    return frames, flows


def evaluate_attack(model, loader, mode, params, label):
    model.eval()
    preds_all, labels_all = [], []
    mae_total, n_batches = 0.0, 0

    with torch.no_grad():
        for frames, flows, labels, density_targets in loader:
            frames, flows = apply_attack(frames, flows, mode, params)
            frames          = frames.to(device)
            flows           = flows.to(device)
            labels          = labels.to(device)
            density_targets = density_targets.to(device)

            logits, density_preds = model(frames, flows)
            density_preds = torch.clamp(density_preds, min=0.0)

            preds = torch.argmax(logits, dim=1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pred_counts = density_preds.sum(dim=[1,2,3]).cpu().numpy()
            true_counts = density_targets.sum(dim=[1,2,3]).cpu().numpy()
            mae_total  += np.mean(np.abs(pred_counts - true_counts))
            n_batches  += 1

    acc = 100 * accuracy_score(labels_all, preds_all)
    mae = mae_total / n_batches

    print(f"\n{'='*60}\n[{ABLATION_CONFIG}] ATTACK : {label}\n{'='*60}")
    print(f"Accuracy  : {acc:.2f}%")
    print(f"Count MAE : {mae:.2f} agents")
    print(classification_report(labels_all, preds_all,
                                target_names=["Normal", "Bottleneck", "Panic"],
                                digits=4, zero_division=0))
    print(confusion_matrix(labels_all, preds_all))
    return acc, mae


print(f"\n{'='*60}\nEXTENDED ROBUSTNESS SUITE — [{ABLATION_CONFIG}]\n{'='*60}")

attacks = [
    ("salt_pepper",   {"prob": 0.02},  "Salt & Pepper (2%)"),
    ("salt_pepper",   {"prob": 0.05},  "Salt & Pepper (5%)"),
    ("fog",           {"intensity": 0.30}, "Fog (30%)"),
    ("fog",           {"intensity": 0.60}, "Fog (60%)"),
    ("frame_dropout", {"prob": 0.20}, "Frame Dropout (20%)"),
    ("frame_dropout", {"prob": 0.40}, "Frame Dropout (40%)"),
    ("fog_gaussian",  {"fog_intensity": 0.30, "gaussian_sigma": 0.05}, "Fog+Gaussian"),
    ("heavy_blur",    {"kernel": 15}, "Heavy Blur (15x15)"),
    ("brightness",    {"factor": 0.5}, "Brightness -50%"),
    ("brightness",    {"factor": 1.5}, "Brightness +50%"),
]

summary = []
for mode, params, label in attacks:
    acc, mae = evaluate_attack(model, test_loader, mode, params, label)
    summary.append((label, acc, mae))

# ── Clean baseline computed live — NEVER hardcoded ────────────
clean_acc_check, clean_mae_check = evaluate_attack(
    model, test_loader, mode=None, params={}, label="Clean baseline (recomputed)"
)

print(f"\n{'='*70}\nEXTENDED ROBUSTNESS SUMMARY — [{ABLATION_CONFIG}]\n{'='*70}")
print(f"{'Attack':<35}  {'Accuracy':>10}  {'Count MAE':>10}")
print("-"*70)
print(f"{'Clean baseline':<35}  {clean_acc_check:>9.2f}%  {clean_mae_check:>9.2f}")
for label, acc, mae in summary:
    print(f"{label:<35}  {acc:>9.2f}%  {mae:>9.2f}")
print("="*70)

# Save full results to disk so all 4 ablation runs can be merged later
import json
ablation_results["extended_attacks"] = {label: {"acc": acc, "mae": mae} for label, acc, mae in summary}
ablation_results["clean_acc_recheck"] = clean_acc_check
ablation_results["clean_mae_recheck"] = clean_mae_check

with open(f"ablation_results_{ABLATION_CONFIG}.json", "w") as f:
    json.dump(ablation_results, f, indent=2)

print(f"\nSaved full results to ablation_results_{ABLATION_CONFIG}.json")


EXTENDED ROBUSTNESS SUITE — [motioncnn_no_aug]

FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents


[motioncnn_no_aug] ATTACK : Salt & Pepper (2%)
Accuracy  : 33.33%
Count MAE : 81.24 agents
              precision    recall  f1-score   support

      Normal     0.0000    0.0000    0.0000        90
  Bottleneck     0.0000    0.0000    0.0000        90
       Panic     0.3333    1.0000    0.5000        90

    accuracy                         0.3333       270
   macro avg     0.1111    0.3333    0.1667       270
weighted avg     0.1111    0.3333    0.1667       270

[[ 0  0 90]
 [ 0  0 90]
 [ 0  0 90]]

FLOW VERIFICATION (idx=0)
  flow std    : 0.1059
  density sum : 114.0 agents


[motioncnn_no_aug] ATTACK : Salt & Pepper (5%)
Accuracy  : 33.33%
Count MAE : 220.66 agents
              precision    recall  f1-score   support

      Normal     0.0000    0.0000    0.0000        90
  Bottleneck     0.0000    0.0000    0.0000        90
       Panic     0.3333   

In [ ]:
# ============================================================
# CELL 10 : PER-SAMPLE VISUALISATION
# ============================================================
#
# Shows the model's prediction on individual test samples.
# For each sample:
#   - Grayscale frame (last frame in window)
#   - Optical flow visualised as HSV colour wheel
#   - Predicted density map
#   - Classification probabilities as bar chart
#   - True label vs predicted label
#
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import random

CLASS_NAMES  = {0: "Normal", 1: "Compressed", 2: "Turbulent"}
CLASS_COLORS = {0: "#2196F3", 1: "#FF9800", 2: "#F44336"}


def flow_to_rgb(flow_np):
    """
    Converts a 2-channel flow array (H,W,2) to an RGB image
    using HSV colour encoding: hue=direction, value=magnitude.
    """
    fx = flow_np[:, :, 0]
    fy = flow_np[:, :, 1]
    magnitude = np.sqrt(fx**2 + fy**2)
    angle     = np.arctan2(fy, fx)          # -pi to pi

    hsv = np.zeros((*magnitude.shape, 3), dtype=np.uint8)
    hsv[:, :, 0] = ((angle + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    mag_norm      = magnitude / (magnitude.max() + 1e-6)
    hsv[:, :, 1] = (mag_norm * 255).astype(np.uint8)
    hsv[:, :, 2] = 255

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def visualise_predictions(model, dataset, n_samples=9, seed=42):
    """
    Visualises model predictions on n_samples test items.
    Samples are drawn to include all three classes (3 per class).
    """
    model.eval()
    random.seed(seed)

    # Gather indices per class
    class_indices = {0: [], 1: [], 2: []}
    for i, w in enumerate(dataset.windows):
        class_indices[w["label"]].append(i)

    # Sample 3 from each class
    selected = []
    for cls in range(3):
        chosen = random.sample(class_indices[cls],
                               min(n_samples // 3, len(class_indices[cls])))
        selected.extend(chosen)
    random.shuffle(selected)
    selected = selected[:n_samples]

    n_cols = 3
    n_rows = len(selected)

    fig = plt.figure(figsize=(18, n_rows * 5))
    fig.patch.set_facecolor("#1a1a2e")

    gs_outer = gridspec.GridSpec(n_rows, 1, hspace=0.08,
                                 left=0.02, right=0.98,
                                 top=0.95, bottom=0.02)

    fig.suptitle(
        "CrowdNet — Per-Sample Predictions on Test Set",
        fontsize=18, color="white", fontweight="bold", y=0.98
    )

    with torch.no_grad():
        for row_idx, sample_idx in enumerate(selected):

            frames, flows, label, density_target = dataset[sample_idx]

            # Forward pass
            frames_in = frames.unsqueeze(0).to(device)
            flows_in  = flows.unsqueeze(0).to(device)
            logits, density_pred = model(frames_in, flows_in)
            density_pred = torch.clamp(density_pred, min=0.0)

            probs     = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            pred_cls  = int(probs.argmax())
            true_cls  = int(label.item())
            correct   = pred_cls == true_cls

            # Extract last frame for display
            last_frame = frames[-1, 0].numpy()       # (H, W)

            # Extract last flow for display
            last_flow  = flows[-1].permute(1, 2, 0).numpy()   # (H, W, 2)
            flow_rgb   = flow_to_rgb(last_flow)

            # Density map
            dens_map   = density_pred.squeeze().cpu().numpy()  # (28, 28)

            # ── Inner grid: 4 panels per row ─────────────────
            gs_inner = gridspec.GridSpecFromSubplotSpec(
                1, 4, subplot_spec=gs_outer[row_idx],
                wspace=0.04, hspace=0
            )

            border_color = "#4CAF50" if correct else "#F44336"

            # ─── Panel 1: Last frame ─────────────────────────
            ax1 = fig.add_subplot(gs_inner[0])
            ax1.imshow(last_frame, cmap="gray", vmin=0, vmax=1)
            ax1.set_title(
                f"Frame (last)\nTrue: {CLASS_NAMES[true_cls]}",
                color="white", fontsize=9, pad=3
            )
            ax1.axis("off")
            for spine in ax1.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(2)

            # ─── Panel 2: Optical flow ───────────────────────
            ax2 = fig.add_subplot(gs_inner[1])
            ax2.imshow(flow_rgb)
            ax2.set_title(
                "Farneback Flow\n(hue=direction, brightness=magnitude)",
                color="white", fontsize=9, pad=3
            )
            ax2.axis("off")

            # ─── Panel 3: Density map ────────────────────────
            ax3 = fig.add_subplot(gs_inner[2])
            im = ax3.imshow(dens_map, cmap="hot", interpolation="bilinear")
            pred_count = float(dens_map.sum())
            true_count = float(density_target.sum().item())
            ax3.set_title(
                f"Predicted Density\nPred cnt: {pred_count:.0f}  True cnt: {true_count:.0f}",
                color="white", fontsize=9, pad=3
            )
            ax3.axis("off")
            plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

            # ─── Panel 4: Class probabilities ────────────────
            ax4 = fig.add_subplot(gs_inner[3])
            ax4.set_facecolor("#16213e")
            colors = [CLASS_COLORS[i] for i in range(3)]
            bars   = ax4.barh(
                [CLASS_NAMES[i] for i in range(3)],
                probs, color=colors, height=0.5
            )
            ax4.set_xlim(0, 1)
            ax4.set_xlabel("Probability", color="white", fontsize=8)
            ax4.tick_params(colors="white", labelsize=8)
            ax4.spines["bottom"].set_color("white")
            ax4.spines["left"].set_color("white")
            ax4.spines["top"].set_visible(False)
            ax4.spines["right"].set_visible(False)
            for bar, prob in zip(bars, probs):
                ax4.text(
                    prob + 0.02, bar.get_y() + bar.get_height() / 2,
                    f"{prob:.3f}", va="center", color="white", fontsize=8
                )
            verdict = "CORRECT" if correct else "WRONG"
            verdict_col = "#4CAF50" if correct else "#F44336"
            ax4.set_title(
                f"Pred: {CLASS_NAMES[pred_cls]}  [{verdict}]",
                color=verdict_col, fontsize=9, fontweight="bold", pad=3
            )

    plt.savefig(
        "crowdnet_predictions_new.png",
        dpi=150, bbox_inches="tight",
        facecolor="#1a1a2e"
    )
    plt.show()
    print("Saved: crowdnet_predictions_new.png")


# ── Run visualisation ─────────────────────────────────────────
visualise_predictions(model, test_dataset, n_samples=9, seed=42)


# ============================================================
# ADDITIONAL: Temporal evolution plot for one sequence
# ============================================================

def visualise_temporal_evolution(model, dataset, class_target=1, seed=7):
    """
    Shows how the model's confidence evolves frame by frame
    across a single sequence, for one sequence of class_target.
    Picks 6 evenly-spaced frames from the sequence window.
    """
    model.eval()
    random.seed(seed)

    # Find a window of the target class
    candidates = [i for i, w in enumerate(dataset.windows)
                  if w["label"] == class_target]
    idx = random.choice(candidates)
    frames, flows, label, density_target = dataset[idx]

    T = frames.shape[0]
    step_indices = np.linspace(0, T - 1, 6, dtype=int)

    fig, axes = plt.subplots(3, 6, figsize=(22, 9))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(
        f"Temporal Evolution — True Class: {CLASS_NAMES[int(label.item())]}",
        fontsize=14, color="white", fontweight="bold"
    )

    with torch.no_grad():
        for col, t in enumerate(step_indices):

            # Run model up to timestep t (truncated sequence)
            frames_in = frames[:t+1].unsqueeze(0).to(device)
            flows_in  = flows[:t+1].unsqueeze(0).to(device)
            logits, _  = model(frames_in, flows_in)
            probs      = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            pred_cls   = int(probs.argmax())

            # Frame
            ax_frame = axes[0, col]
            ax_frame.imshow(frames[t, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax_frame.set_title(f"t={t}", color="white", fontsize=9)
            ax_frame.axis("off")

            # Flow
            ax_flow = axes[1, col]
            flow_np = flows[t].permute(1, 2, 0).numpy()
            ax_flow.imshow(flow_to_rgb(flow_np))
            ax_flow.axis("off")

            # Probabilities
            ax_prob = axes[2, col]
            ax_prob.set_facecolor("#16213e")
            colors = [CLASS_COLORS[i] for i in range(3)]
            ax_prob.bar(
                [CLASS_NAMES[i] for i in range(3)],
                probs, color=colors
            )
            ax_prob.set_ylim(0, 1)
            ax_prob.tick_params(colors="white", labelsize=7)
            ax_prob.set_facecolor("#16213e")
            for spine in ax_prob.spines.values():
                spine.set_color("white")
            ax_prob.set_title(
                f"Pred: {CLASS_NAMES[pred_cls]}",
                color=CLASS_COLORS[pred_cls], fontsize=8, fontweight="bold"
            )

    axes[0, 0].set_ylabel("Frame", color="white", fontsize=9)
    axes[1, 0].set_ylabel("Flow (HSV)", color="white", fontsize=9)
    axes[2, 0].set_ylabel("Class Prob.", color="white", fontsize=9)

    plt.tight_layout()
    plt.savefig(
        "crowdnet_temporal_evolution_new.png",
        dpi=150, bbox_inches="tight",
        facecolor="#1a1a2e"
    )
    plt.show()
    print("Saved: crowdnet_temporal_evolution_new.png")


# Show temporal evolution for one Bottleneck sequence
visualise_temporal_evolution(model, test_dataset, class_target=1, seed=7)
# Show temporal evolution for one Panic sequence
#visualise_temporal_evolution(model, test_dataset, class_target=2, seed=13)

In [9]:
# ============================================================
# FINAL ABLATION COMPARISON TABLE
# Run this once all 4 configs have produced their JSON files
# ============================================================

import json

configs = ["baseline", "baseline_aug", "motioncnn_no_aug", "motioncnn_aug"]
all_results = {}

for cfg in configs:
    try:
        with open(f"ablation_results_{cfg}.json") as f:
            all_results[cfg] = json.load(f)
    except FileNotFoundError:
        print(f"WARNING: ablation_results_{cfg}.json not found yet")

print(f"\n{'='*100}")
print("ABLATION STUDY — FINAL COMPARISON")
print(f"{'='*100}")
print(f"{'Config':<20} {'Clean Acc':>10} {'Clean MAE':>10} {'Shuffle Drop':>13} {'ZeroFlow Drop':>14} {'S&P 5%':>8}")
print("-"*100)

for cfg in configs:
    if cfg not in all_results:
        continue
    r = all_results[cfg]
    sp5 = r["extended_attacks"].get("Salt & Pepper (5%)", {}).get("acc", float("nan"))
    print(f"{cfg:<20} {r['clean_acc_recheck']:>9.2f}% {r['clean_mae_recheck']:>9.2f} "
          f"{r['shuffle_drop']:>12.2f}% {r['zflow_drop']:>13.2f}% {sp5:>7.2f}%")

print("="*100)


ABLATION STUDY — FINAL COMPARISON
Config                Clean Acc  Clean MAE  Shuffle Drop  ZeroFlow Drop   S&P 5%
----------------------------------------------------------------------------------------------------
baseline                 99.63%      2.17        15.56%         66.30%   33.33%
baseline_aug             99.26%      1.63        20.37%         52.22%   99.26%
motioncnn_no_aug         97.04%      1.90         7.78%         61.85%   33.33%
motioncnn_aug            98.15%      1.68        12.22%         64.81%   92.59%
